# 녹화 영상 슬라이싱

서버 PC에서 녹화를 마친 뒤 실행하는 노트북입니다.

- 세션의 영상 4개를 자동으로 찾습니다.
- 네 영상에서 같은 목표 시각의 프레임을 추출합니다.
- 같은 시각에는 같은 파일명을 사용합니다.
- 흐림 점수는 CSV에 기록하지만 프레임 동기화를 유지하기 위해 자동 삭제하지 않습니다.

기본값은 1초마다 한 장입니다.

## 1. 세션과 추출 옵션 설정

SESSION_DIR을 camera_recording 노트북에서 출력된 실제 세션 경로로 수정하세요.

- INTERVAL_SEC: 이미지 간 시간 간격
- START_OFFSET_SEC: 영상 앞부분에서 건너뛸 시간
- END_OFFSET_SEC: 영상 뒷부분에서 제외할 시간
- OUTPUT_DIR: None이면 시간표가 붙은 새 출력 폴더를 자동 생성
- JPEG_QUALITY: 1~100

In [ ]:
from pathlib import Path

SESSION_DIR = Path(
    "/home/syw/Trihouse/dataset/camera_recordings/session_YYYYMMDD_HHMMSS"
)

INTERVAL_SEC = 1.0
START_OFFSET_SEC = 0.0
END_OFFSET_SEC = 0.0
JPEG_QUALITY = 95

# None 권장: 기존 결과를 덮어쓰지 않고 새 폴더 생성
OUTPUT_DIR = None

## 2. 영상 검색과 공통 목표 시각 계산

각 영상은 실제 프레임 수가 조금 다를 수 있으므로 가장 짧은 영상 길이를 공통 종료점으로 사용합니다.
예를 들어 1초 간격이면 모든 카메라에서 0초, 1초, 2초 프레임을 같은 이름으로 저장합니다.

In [ ]:
import math
from pathlib import Path


VIDEO_SUFFIXES = {".mp4", ".mkv", ".avi", ".mov"}


def discover_videos(session_dir: Path) -> list[Path]:
    session_dir = Path(session_dir)
    video_dir = session_dir / "videos"
    search_dir = video_dir if video_dir.is_dir() else session_dir
    if not search_dir.is_dir():
        raise FileNotFoundError(f"영상 폴더가 없습니다: {search_dir}")

    videos = sorted(
        path
        for path in search_dir.iterdir()
        if path.is_file() and path.suffix.lower() in VIDEO_SUFFIXES
    )
    if not videos:
        raise FileNotFoundError(f"영상 파일이 없습니다: {search_dir}")
    return videos


def build_target_times(
    common_duration_s: float,
    interval_s: float,
    start_offset_s: float = 0.0,
    end_offset_s: float = 0.0,
) -> list[float]:
    if interval_s <= 0:
        raise ValueError("INTERVAL_SEC은 0보다 커야 합니다.")
    if common_duration_s <= 0:
        raise ValueError("영상 길이는 0보다 커야 합니다.")
    if start_offset_s < 0 or end_offset_s < 0:
        raise ValueError("시작/종료 오프셋은 0 이상이어야 합니다.")

    end_time = common_duration_s - end_offset_s
    if start_offset_s >= end_time:
        raise ValueError("추출할 수 있는 구간이 없습니다.")

    target_times = []
    index = 0
    while True:
        target_time = start_offset_s + index * interval_s
        if target_time >= end_time - 1e-9:
            break
        target_times.append(round(target_time, 9))
        index += 1
    return target_times

## 3. 영상 정보 확인

이 단계에서는 영상을 수정하지 않습니다. 파일별 FPS, 프레임 수와 길이를 확인하고 공통 추출 시각을 계산합니다.

In [ ]:
import cv2


def probe_video(video_path: Path) -> dict:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"영상을 열 수 없습니다: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    if fps <= 0 or frame_count <= 0:
        raise RuntimeError(
            f"잘못된 영상 정보: {video_path.name}, fps={fps}, frames={frame_count}"
        )

    return {
        "camera_id": video_path.stem,
        "path": video_path,
        "fps": fps,
        "frame_count": frame_count,
        "duration_s": frame_count / fps,
        "width": width,
        "height": height,
    }


VIDEO_PATHS = discover_videos(SESSION_DIR)
VIDEO_INFOS = [probe_video(path) for path in VIDEO_PATHS]

for info in VIDEO_INFOS:
    print(
        f"{info['camera_id']}: {info['width']}x{info['height']}, "
        f"{info['fps']:.2f} FPS, {info['frame_count']} frames, "
        f"{info['duration_s']:.2f}s"
    )

COMMON_DURATION_SEC = min(info["duration_s"] for info in VIDEO_INFOS)
TARGET_TIMES = build_target_times(
    COMMON_DURATION_SEC,
    INTERVAL_SEC,
    START_OFFSET_SEC,
    END_OFFSET_SEC,
)

print(f"\n공통 영상 길이: {COMMON_DURATION_SEC:.2f}초")
print(f"추출 예정: 카메라당 {len(TARGET_TIMES)}장")
print(f"총 이미지 예정: {len(TARGET_TIMES) * len(VIDEO_INFOS)}장")

## 4. 같은 시각의 프레임 추출

각 목표 시각에서 네 영상이 모두 읽힌 경우에만 하나의 프레임 세트로 저장합니다.
한 카메라라도 읽지 못하면 해당 시각 전체를 건너뛰므로 카메라별 파일명이 어긋나지 않습니다.

결과 구조:

    sliced_YYYYMMDD_HHMMSS/
    ├── frames/
    │   ├── pinky_01/
    │   ├── pinky_02/
    │   ├── fixed_01/
    │   └── fixed_02/
    └── frames_manifest.csv

In [ ]:
import csv
from datetime import datetime

import numpy as np


def calculate_blur_score(frame: np.ndarray) -> float:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


if OUTPUT_DIR is None:
    run_name = datetime.now().strftime("sliced_%Y%m%d_%H%M%S")
    RESULT_DIR = SESSION_DIR / run_name
else:
    RESULT_DIR = Path(OUTPUT_DIR)

if RESULT_DIR.exists():
    raise FileExistsError(
        f"출력 폴더가 이미 존재합니다: {RESULT_DIR}\n"
        "OUTPUT_DIR을 바꾸거나 None으로 설정해 새 폴더를 만드세요."
    )

frames_root = RESULT_DIR / "frames"
frames_root.mkdir(parents=True, exist_ok=False)

captures = {}
for info in VIDEO_INFOS:
    camera_id = info["camera_id"]
    (frames_root / camera_id).mkdir()
    cap = cv2.VideoCapture(str(info["path"]))
    if not cap.isOpened():
        raise RuntimeError(f"영상을 열 수 없습니다: {info['path']}")
    captures[camera_id] = cap

manifest_rows = []
saved_sets = 0

try:
    for target_index, target_time_s in enumerate(TARGET_TIMES):
        pending_frames = {}

        for info in VIDEO_INFOS:
            camera_id = info["camera_id"]
            cap = captures[camera_id]
            cap.set(cv2.CAP_PROP_POS_MSEC, target_time_s * 1000.0)
            ok, frame = cap.read()
            if not ok:
                pending_frames = {}
                print(f"건너뜀: {target_time_s:.3f}s에서 {camera_id} 읽기 실패")
                break

            source_frame_index = max(
                int(cap.get(cv2.CAP_PROP_POS_FRAMES)) - 1,
                0,
            )
            pending_frames[camera_id] = {
                "frame": frame,
                "source_frame_index": source_frame_index,
                "blur_score": calculate_blur_score(frame),
            }

        if len(pending_frames) != len(VIDEO_INFOS):
            continue

        filename = f"frame_{saved_sets:06d}_t{target_time_s:010.3f}s.jpg"
        for camera_id, record in pending_frames.items():
            output_path = frames_root / camera_id / filename
            ok = cv2.imwrite(
                str(output_path),
                record["frame"],
                [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY],
            )
            if not ok:
                raise RuntimeError(f"이미지 저장 실패: {output_path}")

            manifest_rows.append(
                {
                    "set_index": saved_sets,
                    "camera_id": camera_id,
                    "target_time_s": target_time_s,
                    "source_frame_index": record["source_frame_index"],
                    "blur_score": record["blur_score"],
                    "relative_path": str(output_path.relative_to(RESULT_DIR)),
                }
            )

        saved_sets += 1
        if saved_sets % 20 == 0:
            print(f"{saved_sets}/{len(TARGET_TIMES)} 프레임 세트 저장")

finally:
    for cap in captures.values():
        cap.release()

manifest_path = RESULT_DIR / "frames_manifest.csv"
with manifest_path.open("w", newline="", encoding="utf-8") as csv_file:
    fieldnames = [
        "set_index",
        "camera_id",
        "target_time_s",
        "source_frame_index",
        "blur_score",
        "relative_path",
    ]
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(manifest_rows)

print(f"\n완료: {saved_sets}개 시각 × {len(VIDEO_INFOS)}개 카메라")
print(f"이미지: {frames_root}")
print(f"목록: {manifest_path}")

## 5. 결과 미리보기

첫 번째로 저장된 시각의 네 카메라 이미지를 나란히 확인합니다.

In [ ]:
import matplotlib.pyplot as plt

camera_dirs = sorted(path for path in frames_root.iterdir() if path.is_dir())
preview_paths = []
for camera_dir in camera_dirs:
    images = sorted(camera_dir.glob("*.jpg"))
    if images:
        preview_paths.append((camera_dir.name, images[0]))

if not preview_paths:
    print("미리볼 이미지가 없습니다.")
else:
    fig, axes = plt.subplots(1, len(preview_paths), figsize=(5 * len(preview_paths), 4))
    axes = np.atleast_1d(axes)
    for axis, (camera_id, image_path) in zip(axes, preview_paths):
        image_bgr = cv2.imread(str(image_path))
        axis.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
        axis.set_title(camera_id)
        axis.axis("off")
    fig.tight_layout()
    plt.show()

## 6. 품질 확인 기준

- 네 카메라 폴더의 이미지 개수가 같은지 확인
- 동일 파일명이 비슷한 순간을 보여주는지 확인
- frames_manifest.csv에서 blur_score가 유난히 낮은 프레임을 육안 검토
- 처음에는 30초 영상과 1초 간격으로 시험
- 정상 동작 후 녹화 시간, FPS와 추출 간격을 늘리거나 줄임